# Customer Analytics Executive Report

This notebook follows the StudyBuild customer analytics booklet. I focused on the questions that are most useful for a short management report: overall performance, advertising location, loyalty customers, inactivity risk, and discount behavior.

The analysis is based on summarized customer profiles, not individual orders. For that reason, the conclusions describe patterns in this dataset and should not be treated as proof of cause and effect.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_FILE = Path("cleaned_customer_data.xlsx")

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Could not find {DATA_FILE}")

df = pd.read_excel(DATA_FILE)
print(f"Loaded {len(df)} customer rows and {len(df.columns)} columns.")

## 1. Data quality checks

Before calculating results, I checked the required columns, dates, missing values, duplicate customer IDs, and the existing quality flag. I kept the flagged return records because `returned_items` counts items while `purchase_count` counts purchases, so the two fields are not directly comparable.

In [ ]:
required_columns = [
    "customer_id", "first_name", "gender", "age", "city", "province",
    "signup_date", "membership_tier", "purchase_count", "avg_order_value",
    "total_spending", "last_purchase_days", "payment_method", "device",
    "discount_used", "returned_items", "satisfaction_score"
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

numeric_columns = [
    "age", "purchase_count", "avg_order_value", "total_spending",
    "last_purchase_days", "returned_items", "satisfaction_score"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

text_columns = [
    "first_name", "gender", "city", "province", "membership_tier",
    "payment_method", "device", "discount_used"
]
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

quality_summary = pd.Series({
    "Rows": len(df),
    "Unique customer IDs": df["customer_id"].nunique(),
    "Duplicate customer IDs": df["customer_id"].duplicated().sum(),
    "Duplicate rows": df.duplicated().sum(),
    "Missing required values": int(df[required_columns].isna().sum().sum()),
    "Flagged rows": int(df.get("data_quality_flag", pd.Series(dtype="object")).notna().sum())
}, name="Value")

display(quality_summary)

## 2. Core customer profile

The modal profile helps describe the current customer base, but it should not be the only target. A smaller group can still be more valuable than the largest group.

In [ ]:
age_bins = [0, 24, 34, 44, 54, 64, np.inf]
age_labels = ["Under 25", "25-34", "35-44", "45-54", "55-64", "65+"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, include_lowest=True)

profile_summary = pd.Series({
    "Customers": df["customer_id"].nunique(),
    "Mean age": round(df["age"].mean(), 1),
    "Median age": df["age"].median(),
    "Largest age group": df["age_group"].value_counts().idxmax(),
    "Most common gender": df["gender"].mode().iloc[0],
    "Most common membership tier": df["membership_tier"].mode().iloc[0],
    "Most common device": df["device"].mode().iloc[0],
    "Most common payment method": df["payment_method"].mode().iloc[0]
}, name="Value")

display(profile_summary)
display(df["age_group"].value_counts(sort=False).rename("Customers"))

## 3. Advertising location

I compared cities using customer count, total and average spending, purchase frequency, order value, and satisfaction. Groups with fewer than three customers are excluded to reduce the effect of unstable averages.

In [ ]:
city_summary = (
    df.groupby(["province", "city"], as_index=False)
    .agg(
        customer_count=("customer_id", "nunique"),
        total_revenue=("total_spending", "sum"),
        avg_customer_spending=("total_spending", "mean"),
        avg_purchase_count=("purchase_count", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        avg_satisfaction=("satisfaction_score", "mean")
    )
)
city_summary = city_summary[city_summary["customer_count"] >= 3].copy()
city_summary = city_summary.sort_values("total_revenue", ascending=False)

display(city_summary.round(2))

chart1 = city_summary.head(10).sort_values("total_revenue")
chart1_labels = chart1["province"] + " - " + chart1["city"]

plt.figure(figsize=(10, 6))
plt.barh(chart1_labels, chart1["total_revenue"], color="#4472C4")
plt.title("Top Cities by Total Customer Spending")
plt.xlabel("Total customer spending")
plt.ylabel("Province - city")
plt.tight_layout()
plt.show()

## 4. Loyalty and inactivity risk

The loyalty score uses the booklet weights: spending 35%, purchase count 30%, recency 20%, and satisfaction 15%. These weights are an analytical choice, not a permanent business rule.

In [ ]:
loyalty_df = df.dropna(subset=[
    "customer_id", "first_name", "purchase_count", "total_spending",
    "last_purchase_days", "satisfaction_score"
]).copy()

for column in ["purchase_count", "total_spending", "satisfaction_score"]:
    minimum = loyalty_df[column].min()
    maximum = loyalty_df[column].max()
    if maximum == minimum:
        loyalty_df[f"{column}_score"] = 1.0
    else:
        loyalty_df[f"{column}_score"] = (loyalty_df[column] - minimum) / (maximum - minimum)

recency_min = loyalty_df["last_purchase_days"].min()
recency_max = loyalty_df["last_purchase_days"].max()
if recency_max == recency_min:
    loyalty_df["recency_score"] = 1.0
else:
    loyalty_df["recency_score"] = 1 - (
        (loyalty_df["last_purchase_days"] - recency_min) / (recency_max - recency_min)
    )

loyalty_df["loyalty_score"] = (
    0.35 * loyalty_df["total_spending_score"]
    + 0.30 * loyalty_df["purchase_count_score"]
    + 0.20 * loyalty_df["recency_score"]
    + 0.15 * loyalty_df["satisfaction_score_score"]
)

top_loyal_customers = loyalty_df.nlargest(10, "loyalty_score")[[
    "customer_id", "first_name", "membership_tier", "purchase_count",
    "total_spending", "last_purchase_days", "satisfaction_score", "loyalty_score"
]]

display(top_loyal_customers.round({"total_spending": 2, "loyalty_score": 4}))

In [ ]:
risk_df = df.dropna(subset=[
    "customer_id", "first_name", "total_spending", "purchase_count", "last_purchase_days"
]).copy()

spending_threshold = risk_df["total_spending"].quantile(0.75)
purchase_threshold = risk_df["purchase_count"].median()
inactivity_threshold = risk_df["last_purchase_days"].quantile(0.75)

risk_df["at_risk"] = (
    (risk_df["total_spending"] >= spending_threshold)
    & (risk_df["purchase_count"] >= purchase_threshold)
    & (risk_df["last_purchase_days"] >= inactivity_threshold)
)

at_risk_customers = risk_df.loc[risk_df["at_risk"], [
    "customer_id", "first_name", "membership_tier", "purchase_count",
    "total_spending", "last_purchase_days", "satisfaction_score"
]].sort_values(["total_spending", "last_purchase_days"], ascending=[False, False])

display(at_risk_customers)

plt.figure(figsize=(10, 6))
plt.scatter(
    risk_df.loc[~risk_df["at_risk"], "last_purchase_days"],
    risk_df.loc[~risk_df["at_risk"], "total_spending"],
    alpha=0.45,
    label="Other customers"
)
plt.scatter(
    risk_df.loc[risk_df["at_risk"], "last_purchase_days"],
    risk_df.loc[risk_df["at_risk"], "total_spending"],
    marker="X",
    s=110,
    color="#C00000",
    label="Valuable customers at risk"
)
plt.axvline(inactivity_threshold, linestyle="--", color="grey")
plt.axhline(spending_threshold, linestyle="--", color="grey")
plt.title("Valuable Customers at Risk of Becoming Inactive")
plt.xlabel("Days since last purchase")
plt.ylabel("Total customer spending")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Discount behavior

This section is descriptive. A difference between the discount groups does not mean that discounts caused the difference.

In [ ]:
summary_by_discount = (
    df.groupby("discount_used", as_index=False)
    .agg(
        customer_count=("customer_id", "nunique"),
        avg_total_spending=("total_spending", "mean"),
        avg_purchase_count=("purchase_count", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        avg_returned_items=("returned_items", "mean"),
        avg_satisfaction=("satisfaction_score", "mean")
    )
)

display(summary_by_discount.round(2))

discount_chart = summary_by_discount.set_index("discount_used")[
    ["avg_total_spending", "avg_returned_items"]
]
discount_chart.plot(kind="bar", subplots=True, legend=False, figsize=(8, 7), color=["#70AD47"])
plt.suptitle("Spending and Returned Items by Discount Use")
plt.xlabel("Discount used")
plt.tight_layout()
plt.show()

device_summary = df.groupby("device")["total_spending"].agg(["count", "mean"]).sort_values("mean", ascending=False)
payment_summary = df.groupby("payment_method")["total_spending"].agg(["count", "mean"]).sort_values("mean", ascending=False)

display(device_summary.round(2))
display(payment_summary.round(2))
print("This analysis does not establish a causal relationship.")

## 6. Executive summary

The report stays within the booklet limit of six KPIs and three charts.

In [ ]:
kpi_summary = pd.Series({
    "Unique customers": df["customer_id"].nunique(),
    "Total customer spending": df["total_spending"].sum(),
    "Average spending per customer": df["total_spending"].mean(),
    "Average purchase count": df["purchase_count"].mean(),
    "Average satisfaction": df["satisfaction_score"].mean(),
    "Total returned items": df["returned_items"].sum()
}, name="Value")

display(kpi_summary.round(2))

### Management recommendations

1. **Advertising → Evidence → Action:** Mashhad has the highest total customer spending and a strong average purchase count. Run a measured advertising pilot there, but monitor satisfaction because its average score is only moderate. Isfahan is a useful smaller alternative because it has the highest average spending and satisfaction.

2. **Loyalty → Evidence → Action:** The ranked list includes Bronze, Gold, Silver, and VIP customers, showing that membership tier alone does not identify the strongest customers. Invite the top ten based on the combined score and tailor the reward for customers with low satisfaction.

3. **Inactivity risk → Evidence → Action:** Three high-value customers meet the inactivity-risk definition. Contact customers 1010 and 1053 first for service recovery because their satisfaction score is 1, then send customer 1035 a time-limited win-back offer. Measure whether they purchase again within 30 days.

### Limitations

- Each row is a summarized customer profile, so order-level behavior cannot be examined.
- The reference date for `last_purchase_days` is unknown. The analysis identifies inactivity risk, not confirmed churn.
- The true return rate cannot be calculated because the number of purchased items is unavailable.
- City thresholds, loyalty weights, and risk cutoffs are transparent analytical choices and could be tested with other assumptions.
- Discount results are associations only and do not prove a causal effect.

Excel export is intentionally not included in this notebook revision, so running the analysis does not overwrite the existing result workbook.